# 4_figures/02 — Generate manuscript figure data

Runs the seven `figures/prep/figureN.py` modules with the active **Python kernel**. Each reads from
`DATA_PATH` and writes CSVs to `SURV_PATH/results/figure_data/`, which the R rendering tier
(`03_render_figures.Rmd`) plots from.

- **Code lookups (R)**: [01_code_lookups.Rmd](01_code_lookups.Rmd) — one-time bootstrap.
- **Prep tier**: this notebook (Python).
- **Render tier**: [03_render_figures.Rmd](03_render_figures.Rmd) (R Markdown).

### Incremental by default

A module is **skipped when all of its output CSVs already exist**, so a re-run regenerates only what
is actually missing. Most of these modules re-read large parquets and refit clustering or scoring
work to produce a handful of small CSVs, so skipping a satisfied module is the difference between
seconds and many minutes.

The skip is deliberately coarse — whole module, not per-CSV — because these modules share
intermediates internally (`figure2`'s `stage_vs_risk_df` feeds four outputs, `figure4`'s clustering
feeds five), so producing one missing CSV costs essentially the same as producing all of them.

**The check is presence, not freshness.** It cannot tell a stale CSV from a current one. After
anything upstream changes — a cohort rebuild, new trajectories from `2_models/03`, new metrics from `2_models/02`, a
re-run biomarker pipeline — set `FORCE` for the affected modules, or `REGENERATE_ALL = True`.

**Prerequisite:** run `4_figures/01` once before the first run of this notebook, and again after a cohort rebuild. This notebook
does not invoke R — absent lookups make figure2 fall back to raw codes and log the miss counts
rather than failing; the setup cell tells you which case you are in.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "figures").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import CODE_PATH, FIGURE_DATA_DIR  # noqa: E402

print(f"repo root:     {REPO_ROOT}")
print(f"figure data: {FIGURE_DATA_DIR}")

# Preflight: figure2's phecode labels come from the lookups 4_figures/01 builds. Absent lookups are a
# degraded run, not a failure — say so loudly rather than silently.
_missing = [name for name in ("icd10_to_phecode_mapping.csv", "phecode_descriptions.csv")
            if not os.path.exists(os.path.join(CODE_PATH, name))]
if _missing:
    print("\nWARNING: missing code lookups: " + ", ".join(_missing)
          + "\n  figure2 will fall back to raw codes and disable cross-scheme event dedup."
          + "\n  Run 01_code_lookups.Rmd for manuscript-quality labels.")
else:
    print("code lookups: present (4_figures/01 has run).")

## Configuration

`OUTPUTS` mirrors the `save_figure_data` calls in each module. If a module gains or loses an output,
update it here too — an entry that overstates a module's outputs makes it always re-run; one that
understates makes it skip while a CSV is still missing.

In [ ]:
# module -> the figure-data CSVs its main() writes
OUTPUTS: dict[str, list[str]] = {
    "figure0": [
        "fig0_data_availability.csv", "fig0_availability_combinations.csv",
    ],
    "figure1": [
        "fig1_endpoint_counts.csv", "fig1_cancer_type_counts.csv", "fig1_stage_counts.csv",
        "fig1_treatment_counts.csv", "fig1_notes_per_patient.csv",
    ],
    "figure2": [
        "fig2_full_cohort_metrics.csv", "fig2_within_vs_pan_cancer.csv",
        "fig2_within_vs_pan_treatment.csv", "fig2_km_tertiles.csv",
        "fig2_km_stage_vs_risk.csv", "fig2_stage_vs_risk_cindex.csv",
        "fig2_stage_vs_risk_cindex_by_stage.csv", "fig2_stage_vs_risk_auc.csv",
        *(f"fig2_scheme_delta_topk_{m}.csv" for m in ("cindex", "auc")),
        *(f"fig2_scheme_event_km_{m}.csv" for m in ("cindex", "auc")),
    ],
    "figure2_anchor": [
        "fig2_anchor_sensitivity.csv", "fig2_anchor_cohort_overlap.csv",
    ],
    "figure3": [
        "fig3_modality_cindex.csv",
        *(f"fig3_modality_avg_rank_{t}.csv" for t in ("cindex", "auc")),
        *(f"fig3_modality_ranks_long_{t}.csv" for t in ("cindex", "auc")),
        "fig3_joint_betas.csv", "fig3_risk_score_corr.csv",
    ],
    "figure4": [
        "fig4_trajectories_heatmap.csv", "fig4_km_data.csv", "fig4_cluster_severity.csv",
        "fig4_group_trajectories.csv", "fig4_slope_by_stage.csv", "fig4_silhouette.csv",
    ],
    "figure5": [
        "fig5_ps_predictions.csv", "fig5_robust_hits.csv", "fig5_km_top_hit.csv",
        "fig5_km_examples.csv", "fig5_top_hit_meta.csv", "fig5_love_smd.csv",
        "fig5_forest_headline.csv",
    ],
}

# Extra CLI args per module, e.g. {"figure4": ["--decay", "0.1"]}.
EXTRA_ARGS: dict[str, list[str]] = {}

# --- Run controls ---
REGENERATE_ALL = False        # True -> ignore existing outputs entirely
FORCE: set[str] = set()       # e.g. {"figure4"} to rebuild just that one
ONLY: set[str] = set()        # non-empty -> restrict the run to these modules

print(f"regenerate all: {REGENERATE_ALL}")
print(f"force:          {', '.join(sorted(FORCE)) or 'none'}")
print(f"only:           {', '.join(sorted(ONLY)) or 'all modules'}")

## Plan

What each module would do, before anything runs. Modules with every output present are skipped;
partial ones re-run in full and rewrite all of their CSVs.

In [ ]:
def missing_outputs(module: str) -> list[str]:
    return [name for name in OUTPUTS[module]
            if not os.path.exists(os.path.join(FIGURE_DATA_DIR, name))]


modules = [m for m in OUTPUTS if not ONLY or m in ONLY]
plan = []

for module in modules:
    absent = missing_outputs(module)
    if REGENERATE_ALL:
        reason = "forced (REGENERATE_ALL)"
    elif module in FORCE:
        reason = "forced"
    elif not absent:
        reason = "skip"
    elif len(absent) == len(OUTPUTS[module]):
        reason = f"run — no outputs present ({len(absent)})"
    else:
        reason = f"run — {len(absent)}/{len(OUTPUTS[module])} outputs missing"
    plan.append((module, reason != "skip", absent, reason))

print(f"{'module':<16} {'action':<8} detail")
for module, will_run, absent, reason in plan:
    print(f"{module:<16} {'RUN' if will_run else 'skip':<8} {reason}")
    if will_run and absent and len(absent) < len(OUTPUTS[module]):
        print(f"{'':<25} missing: {', '.join(absent)}")

n_run = sum(1 for _, will_run, _, _ in plan if will_run)
print(f"\n{n_run} module(s) to run, {len(plan) - n_run} skipped")
if not n_run:
    print("Nothing to do — all figure data present. Set REGENERATE_ALL or FORCE to rebuild.")

## Run

Each module is `python -m figures.prep.<module>` with `cwd` set to the repo root. Output streams straight
through. A failure does **not** stop the queue — the modules are independent (they share input data,
not each other's outputs), so a broken figure5 says nothing about figure2. Everything is reported at
the end.

In [ ]:
results = []

for module, will_run, _absent, reason in plan:
    if not will_run:
        print(f"\n=== {module}: skipped ({reason}) ===")
        results.append((module, "skipped", 0.0))
        continue

    print(f"\n{'=' * 78}\n=== figures.prep.{module}  [{reason}]\n{'=' * 78}", flush=True)
    started = time.perf_counter()
    proc = subprocess.run(
        [sys.executable, "-m", f"figures.prep.{module}", *EXTRA_ARGS.get(module, [])],
        cwd=str(REPO_ROOT),
    )
    elapsed = time.perf_counter() - started
    status = "ok" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
    print(f"\n[{module}] {status} in {elapsed / 60:.1f} min")
    results.append((module, status, elapsed))

print(f"\n{'=' * 78}\n=== Run summary ===")
for module, status, elapsed in results:
    print(f"  {module:<16} {status}" + (f"  ({elapsed / 60:.1f} min)" if elapsed else ""))

## Verify

The end state on disk. A module that reported `ok` but still has missing outputs wrote fewer CSVs
than `OUTPUTS` claims — either the module changed or the list here is stale.

Zero-row CSVs are called out separately: `save_figure_data` writes them deliberately on
data-not-available paths and only logs a warning, and the R tier turns one into an empty tibble and
renders a placeholder panel without complaint. On disk they are indistinguishable from a real
result, so they are worth seeing here.

In [ ]:
import polars as pl

all_missing, empty = [], []

for module in modules:
    absent = missing_outputs(module)
    present = [n for n in OUTPUTS[module] if n not in absent]
    all_missing.extend(absent)

    for name in present:
        try:
            if pl.read_csv(os.path.join(FIGURE_DATA_DIR, name)).height == 0:
                empty.append(name)
        except Exception as exc:
            empty.append(f"{name} (unreadable: {type(exc).__name__})")

    flag = "ok " if not absent else "INCOMPLETE"
    print(f"[{flag:<10}] {module:<16} {len(present)}/{len(OUTPUTS[module])} outputs present")
    for name in absent:
        print(f"{'':<13} missing: {name}")

print(f"\n{sum(len(v) for v in OUTPUTS.values()) - len(all_missing)}"
      f"/{sum(len(v) for v in OUTPUTS.values())} figure-data CSVs present")

if empty:
    print(f"\n{len(empty)} file(s) with 0 rows — these render as placeholder panels:")
    for name in empty:
        print(f"  {name}")

failed = [m for m, status, _ in results if status.startswith("FAILED")]
if failed:
    print(f"\n{len(failed)} module(s) failed: {', '.join(failed)}")
elif all_missing:
    print("\nSome outputs are still missing — see above before running 4_figures/03.")
else:
    print("\nAll figure data present. Now run 03_render_figures.Rmd to plot.")